# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Little-Master-Umer/FlyRank_ML_Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

My chosen lane is Refresh / Content Opportunity Scoring.

## Rule

I will prioritize pages for refresh when they are both **stale** and have **meaningful search demand**. The baseline score combines `days_since_last_update` and `search_volume`. A higher score means the page is a stronger candidate for review.

This is a simple decision-support baseline, not a prediction of future performance.

## Reason codes

* `STALE_HIGH_DEMAND` — the page is stale and has relatively high search demand.
* `STALE_MEDIUM_DEMAND` — the page is stale and has moderate search demand.
* `HIGH_DEMAND_NOT_STALE` — the page has search demand but does not appear sufficiently stale.
* `LOW_PRIORITY` — the page has low demand and/or is not sufficiently stale.

## Action labels

* `REFRESH` — prioritize the page for a refresh review.
* `REVIEW` — consider the page for review, but with lower priority.
* `HOLD` — do not prioritize it using this baseline.

The rule uses only information available in the current data window. It does not use future outcomes, product flags, or label-derived inputs.


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
from pathlib import Path
import os

# Clone the repository to get the data
repo_url = "https://github.com/Little-Master-Umer/FlyRank_ML_Internship"
repo_name = repo_url.split('/')[-1]

if not Path(repo_name).exists():
    print(f"Cloning {repo_url}...")
    os.system(f"git clone {repo_url}")
    print("Repository cloned.")
else:
    print("Repository already exists.")

# Adjust the data path after cloning
data_path = Path(f"{repo_name}/data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(data_path)

print("Rows:", len(df))
print("Columns:", len(df.columns))

# Check that the two signals exist
required_cols = ["content_id", "days_since_last_update", "search_volume"]

missing = [c for c in required_cols if c not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Keep only rows with usable signal values
work = df[required_cols].copy()

work["days_since_last_update"] = pd.to_numeric(
    work["days_since_last_update"], errors="coerce"
)

work["search_volume"] = pd.to_numeric(
    work["search_volume"], errors="coerce"
)

work = work.dropna(
    subset=["content_id", "days_since_last_update", "search_volume"]
).copy()

# Robust thresholds based on the observed data.
# This avoids arbitrary fixed numbers and keeps the baseline transparent.
stale_threshold = work["days_since_last_update"].median()
high_demand_threshold = work["search_volume"].quantile(0.75)
medium_demand_threshold = work["search_volume"].median()

print("Stale threshold:", stale_threshold)
print("High-demand threshold:", high_demand_threshold)
print("Medium-demand threshold:", medium_demand_threshold)

# Normalize the two signals to 0-1 using percentile ranks.
# Higher = stronger refresh opportunity.
work["staleness_score"] = work["days_since_last_update"].rank(
    pct=True, method="average"
)

work["demand_score"] = work["search_volume"].rank(
    pct=True, method="average"
)

# Baseline score: equal weight to staleness and demand.
work["score"] = (
    0.50 * work["staleness_score"]
    + 0.50 * work["demand_score"]
)

# Reason codes
work["reason_code"] = np.select(
    [
        (work["days_since_last_update"] >= stale_threshold)
        & (work["search_volume"] >= high_demand_threshold),

        (work["days_since_last_update"] >= stale_threshold)
        & (work["search_volume"] >= medium_demand_threshold),

        (work["days_since_last_update"] < stale_threshold)
        & (work["search_volume"] >= high_demand_threshold),
    ],
    [
        "STALE_HIGH_DEMAND",
        "STALE_MEDIUM_DEMAND",
        "HIGH_DEMAND_NOT_STALE",
    ],
    default="LOW_PRIORITY"
)

# Action labels
work["action"] = np.select(
    [
        work["reason_code"].eq("STALE_HIGH_DEMAND"),
        work["reason_code"].eq("STALE_MEDIUM_DEMAND"),
        work["reason_code"].eq("HIGH_DEMAND_NOT_STALE"),
    ],
    [
        "REFRESH",
        "REVIEW",
        "HOLD",
    ],
    default="HOLD"
)

# Rank highest score first
queue = work.sort_values(
    ["score", "content_id"],
    ascending=[False, True]
).reset_index(drop=True)

queue.insert(0, "rank", range(1, len(queue) + 1))

# Write required output
output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

queue.to_csv(output_path, index=False)

print("\nRanked queue created.")
print("Output:", output_path)
print("Rows:", len(queue))

display(queue.head(20))

Cloning https://github.com/Little-Master-Umer/FlyRank_ML_Internship...
Repository cloned.
Rows: 30000
Columns: 44
Stale threshold: 20.0
High-demand threshold: 20.0
Medium-demand threshold: 10.0

Ranked queue created.
Output: work/outputs/baseline_action_score.csv
Rows: 27532


,rank,content_id,days_since_last_update,search_volume,staleness_score,demand_score,score,reason_code,action
0,1,content_a31e10779c01,144,3600.0,0.995333,0.992391,0.993862,STALE_HIGH_DEMAND,REFRESH
1,2,content_bbca724138f2,236,1600.0,0.999310,0.983783,0.991546,STALE_HIGH_DEMAND,REFRESH
2,3,content_40e140ba2934,231,720.0,0.999128,0.970235,0.984681,STALE_HIGH_DEMAND,REFRESH
3,4,content_24abafed9707,231,480.0,0.999128,0.959302,0.979215,STALE_HIGH_DEMAND,REFRESH
4,5,content_23e958c54c78,144,480.0,0.995333,0.959302,0.977317,STALE_HIGH_DEMAND,REFRESH
5,6,content_29ec1008c834,151,320.0,0.996277,0.945827,0.971052,STALE_HIGH_DEMAND,REFRESH
6,7,content_c3dd69918c8c,151,320.0,0.996277,0.945827,0.971052,STALE_HIGH_DEMAND,REFRESH
7,8,content_6efb8fa48ebe,151,210.0,0.996277,0.929355,0.962816,STALE_HIGH_DEMAND,REFRESH
8,9,content_0cc405838fc5,144,210.0,0.995333,0.929355,0.962344,STALE_HIGH_DEMAND,REFRESH
9,10,content_17e6b2ba4b08,144,170.0,0.995333,0.919821,0.957577,STALE_HIGH_DEMAND,REFRESH


## 2. Build the ranked queue (writes the CSV)

its done in above section.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = queue.head(20).copy()

def confidence_note(row):
    if row["reason_code"] == "STALE_HIGH_DEMAND":
        return "Higher confidence: both baseline signals support refresh."
    elif row["reason_code"] == "STALE_MEDIUM_DEMAND":
        return "Moderate confidence: stale signal supports review, demand is less strong."
    elif row["reason_code"] == "HIGH_DEMAND_NOT_STALE":
        return "Lower confidence: demand is strong, but staleness is not."
    else:
        return "Low confidence: limited support from the two baseline signals."

def wrong_if(row):
    if row["reason_code"] == "STALE_HIGH_DEMAND":
        return "Wrong if the page was recently improved or its demand is not relevant to the page."
    elif row["reason_code"] == "STALE_MEDIUM_DEMAND":
        return "Wrong if the page remains effective despite being old, or demand is too weak to justify work."
    elif row["reason_code"] == "HIGH_DEMAND_NOT_STALE":
        return "Wrong if demand does not translate into a refresh opportunity."
    else:
        return "Wrong if important page context is missing from these two signals."

top20_review = top20[
    [
        "rank",
        "content_id",
        "score",
        "reason_code",
        "action",
        "days_since_last_update",
        "search_volume"
    ]
].copy()

top20_review["confidence_note"] = top20.apply(confidence_note, axis=1)
top20_review["what_would_make_it_wrong"] = top20.apply(wrong_if, axis=1)

display(top20_review)

,rank,content_id,score,reason_code,action,days_since_last_update,search_volume,confidence_note,what_would_make_it_wrong
0,1,content_a31e10779c01,0.993862,STALE_HIGH_DEMAND,REFRESH,144,3600.0,Higher confidence: both baseline signals suppo...,Wrong if the page was recently improved or its...
1,2,content_bbca724138f2,0.991546,STALE_HIGH_DEMAND,REFRESH,236,1600.0,Higher confidence: both baseline signals suppo...,Wrong if the page was recently improved or its...
2,3,content_40e140ba2934,0.984681,STALE_HIGH_DEMAND,REFRESH,231,720.0,Higher confidence: both baseline signals suppo...,Wrong if the page was recently improved or its...
3,4,content_24abafed9707,0.979215,STALE_HIGH_DEMAND,REFRESH,231,480.0,Higher confidence: both baseline signals suppo...,Wrong if the page was recently improved or its...
4,5,content_23e958c54c78,0.977317,STALE_HIGH_DEMAND,REFRESH,144,480.0,Higher confidence: both baseline signals suppo...,Wrong if the page was recently improved or its...
5,6,content_29ec1008c834,0.971052,STALE_HIGH_DEMAND,REFRESH,151,320.0,Higher confidence: both baseline signals suppo...,Wrong if the page was recently improved or its...
6,7,content_c3dd69918c8c,0.971052,STALE_HIGH_DEMAND,REFRESH,151,320.0,Higher confidence: both baseline signals suppo...,Wrong if the page was recently improved or its...
7,8,content_6efb8fa48ebe,0.962816,STALE_HIGH_DEMAND,REFRESH,151,210.0,Higher confidence: both baseline signals suppo...,Wrong if the page was recently improved or its...
8,9,content_0cc405838fc5,0.962344,STALE_HIGH_DEMAND,REFRESH,144,210.0,Higher confidence: both baseline signals suppo...,Wrong if the page was recently improved or its...
9,10,content_17e6b2ba4b08,0.957577,STALE_HIGH_DEMAND,REFRESH,144,170.0,Higher confidence: both baseline signals suppo...,Wrong if the page was recently improved or its...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("## Weak-pick review")

# Show lower-ranked examples that received a non-HOLD action.
weak_candidates = queue[
    queue["action"].isin(["REFRESH", "REVIEW"])
].tail(10)

display(
    weak_candidates[
        [
            "rank",
            "content_id",
            "score",
            "reason_code",
            "action",
            "days_since_last_update",
            "search_volume"
        ]
    ]
)

print("\n## Leakage check")

# Columns used by this baseline
baseline_inputs = [
    "days_since_last_update",
    "search_volume"
]

print("Inputs used:", baseline_inputs)

# Explicitly check for suspicious label/product-style columns
suspicious_terms = [
    "label",
    "target",
    "flag",
    "product",
    "future",
    "outcome"
]

suspicious_columns = [
    c for c in df.columns
    if any(term in c.lower() for term in suspicious_terms)
]

print("Potentially suspicious columns found:")
print(suspicious_columns)

print("\nBaseline inputs are limited to current-window staleness and search demand.")
print("No future outcome or label is used to calculate the score.")
print("No product flag is used to calculate the score.")

## Weak-pick review


,rank,content_id,score,reason_code,action,days_since_last_update,search_volume
19290,19291,content_ff65884e750a,0.426622,STALE_MEDIUM_DEMAND,REVIEW,20,10.0
19291,19292,content_ff7519d55881,0.426622,STALE_MEDIUM_DEMAND,REVIEW,20,10.0
19292,19293,content_ff7f46bd2c17,0.426622,STALE_MEDIUM_DEMAND,REVIEW,20,10.0
19293,19294,content_ff9136889277,0.426622,STALE_MEDIUM_DEMAND,REVIEW,20,10.0
19294,19295,content_ffaaab8d492c,0.426622,STALE_MEDIUM_DEMAND,REVIEW,20,10.0
19295,19296,content_ffc060f09c4c,0.426622,STALE_MEDIUM_DEMAND,REVIEW,20,10.0
19296,19297,content_ffc437abf31c,0.426622,STALE_MEDIUM_DEMAND,REVIEW,20,10.0
19297,19298,content_ffc464755d07,0.426622,STALE_MEDIUM_DEMAND,REVIEW,20,10.0
19298,19299,content_ffca1716a06b,0.426622,STALE_MEDIUM_DEMAND,REVIEW,20,10.0
19299,19300,content_ffd46c8adbbf,0.426622,STALE_MEDIUM_DEMAND,REVIEW,20,10.0



## Leakage check
Inputs used: ['days_since_last_update', 'search_volume']
Potentially suspicious columns found:
[]

Baseline inputs are limited to current-window staleness and search demand.
No future outcome or label is used to calculate the score.
No product flag is used to calculate the score.


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.